In [1]:
# Standard library imports
import os
import sys
import json
import time
import logging
import random

# Third-party imports
import pandas as pd
import numpy
import regex as re
from tqdm import tqdm
import chromadb
from chromadb.utils import embedding_functions

from hipporag import HippoRAG
from hipporag.utils.config_utils import BaseConfig



# Add parent directory to path to import local scripts folder
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Local imports
from scripts.config import COHERE_API_KEY, DATA_DIR, IMAGE_DIR, HIPPORAG_DIR, EMBEDDINGS_1887_COHERE
SH_PATH = os.path.join(DATA_DIR, "questions_singlehop","questions_generated_progressive.jsonl")
from scripts.utils.chroma_utils import query_collection, list_collections, load_chroma_collection
from scripts.utils.json_reading import read_jsonl_basic

# Set environment variable for HippoRAG (uses OpenAI client internally)
os.environ['OPENAI_API_KEY'] = COHERE_API_KEY

# Load collection
from scripts.config import COHERE_API_KEY, EMBEDDINGS_1887_COHERE, HIPPORAG_DIR_MH
from scripts.utils.chroma_utils import load_chroma_collection
from scripts.utils.json_reading import read_jsonl_basic
from scripts.utils.models import load_classifier_model


from scripts.retrieval_utils import (
    RAGRetriever_all_collections_with_classification,
    evaluate_and_print_metrics,
    apply_no_rerouting,
    apply_query_rerouting,
    apply_forced_multicollection,
    apply_forced_equal_from_all_collections,
)
from scripts.run_multi_hop_rag_experiment import perform_multi_hop_retrieval

/home/apellet/anaconda3/envs/hipporag_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-12 10:57:12,227	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


/home/apellet/Desktop/ORDER_EMNLP2026


In [3]:
# Questions files configuration - modify this section to add/remove question types
QUESTION_DATASETS = [
    {
        'name': 'Generic_LeGaulois_Debats',
        'file': 'MH_Generic_legaulois_v1.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': 'Les Débats',
        'question_type': 'MH - Generic'
    },
    {
        'name': 'Generic_LIntransigeant_Debats',
        'file': 'generated_questions_lintransigeant_all.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': "L'Intransigeant",
        'source2_name': "Les Débats",
        'question_type': 'MH - Generic'
    },
    {
        'name': 'BridgeEntity_LIntransigeant_Debats',
        'file': 'MH_BridgeEntity_lintransigeant_v2.jsonl',
        'column_mapping': {
            'id_1': 'journal_id',
            'id_2': 'debat_id',
            'doc_1': 'journal_doc',
            'doc_2': 'debat_doc',
            'metadata_1': 'journal_metadata',
            'metadata_2': 'debat_metadata'
        },
        'source1_name': "L'Intransigeant",
        'source2_name': 'Les Débats',
        'question_type': 'MH - BridgeEntity'
    },
    {
        'name': 'Comparative_LeGaulois_LIntransigeant',
        'file': 'MH_Comparative_legaulois_lintransigeant_v3.jsonl',
        'column_mapping': {
            'id_1': 'legaulois_id',
            'id_2': 'lintransigeant_id',
            'doc_1': 'legaulois_doc',
            'doc_2': 'lintransigeant_doc',
            'metadata_1': 'legaulois_metadata',
            'metadata_2': 'lintransigeant_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': "L'Intransigeant",
        'question_type': 'MH - Comparative'
    },
    {
        'name': 'Generic_LeGaulois_LIntransigeant',
        'file': 'MH_Generic_legaulois_lintransigeant_v3.jsonl',
        'column_mapping': {
            'id_1': 'legaulois_id',
            'id_2': 'lintransigeant_id',
            'doc_1': 'legaulois_doc',
            'doc_2': 'lintransigeant_doc',
            'metadata_1': 'legaulois_metadata',
            'metadata_2': 'lintransigeant_metadata'
        },
        'source1_name': 'Le Gaulois',
        'source2_name': "L'Intransigeant",
        'question_type': 'MH - Generic'
    }
]

def create_questions_dataset(jsonl_path, column_mapping, source1_name='Source 1', 
                            source2_name='Source 2', question_type='Generic'):
    """
    Clean and prepare the multi-hop questions dataset
    
    Args:
        jsonl_path: Path to the JSONL file
        column_mapping: Dict with keys 'id_1', 'id_2', 'doc_1', 'doc_2', 'metadata_1', 'metadata_2'
        source1_name: Name for source 1
        source2_name: Name for source 2
        question_type: Type of questions (e.g., 'MH - Generic', 'MH - BridgeEntity', etc.)
    
    Returns:
        DataFrame with prepared questions
    """
    entries = []
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)
            if isinstance(entry.get('questions'), str):
                entry['questions'] = json.loads(entry['questions'])
            entries.append(entry)
    
    df = pd.DataFrame(entries)
    
    # Create expanded dataset with one row per question
    question_rows = []
    
    for i, row in df.iterrows():
        base_row = row.drop('questions').to_dict()
        
        # Remove original columns to avoid confusion
        for col in column_mapping.values():
            base_row.pop(col, None)
        
        for q_idx, q in enumerate(row["questions"]["questions"]):
            question_row = base_row.copy()
            
            # Handle case where id_2 might be a list (take first element)
            id_2_value = row[column_mapping['id_2']]
            if isinstance(id_2_value, list):
                id_2_value = id_2_value[0] if len(id_2_value) > 0 else None
            
            if id_2_value is None:
                continue  # Skip if no valid id_2
            
            question_row.update({
                'question_index': q_idx,
                'question': q["question"],
                'llm_answer': q.get("answer", ""),
                'question_type': question_type,
                'id_1': row[column_mapping['id_1']],
                'id_2': id_2_value,
                'source_1': source1_name,
                'source_2': source2_name,
                'doc_1': row.get(column_mapping['doc_1'], ''),
                'doc_2': row.get(column_mapping['doc_2'], ''),
                'metadata_1': row.get(column_mapping.get('metadata_1'), {}),
                'metadata_2': row.get(column_mapping.get('metadata_2'), {}),
                'gold_ids': [row[column_mapping['id_1']], id_2_value]
            })
            question_rows.append(question_row)
    
    questions_df = pd.DataFrame(question_rows)
    print(f"Created dataset with {len(questions_df)} questions from {len(df)} original entries")
    return questions_df

# ========================================================================
# Step 2: Load and prepare all question datasets
# ========================================================================
print("\nStep 2: Loading and preparing question datasets...")

all_questions = []

for dataset_config in QUESTION_DATASETS:
    print(f"\n  Processing: {dataset_config['name']}")
    jsonl_path = os.path.join(DATA_DIR, dataset_config['file'])
    
    if not os.path.exists(jsonl_path):
        print(f"    WARNING: File not found - {jsonl_path}")
        continue
    
    df_questions = create_questions_dataset(
        jsonl_path=jsonl_path,
        column_mapping=dataset_config['column_mapping'],
        source1_name=dataset_config['source1_name'],
        source2_name=dataset_config['source2_name'],
        question_type=dataset_config['question_type']
    )
    
    all_questions.append(df_questions)
    print(f"    Loaded {len(df_questions)} questions")

# Combine all question datasets
df_questions = pd.concat(all_questions, ignore_index=True)
df_questions.loc[:,"unique_index"] = df_questions.apply(lambda row: f"{row['source_1']}_{row['source_2']}_{row['question_type']}_{row['id_1']}_{row['id_2']}_{row['question_index']}", axis=1)
df_questions.loc[:,"id_1"] = df_questions["source_1"] + "_" + df_questions["id_1"].astype(str)
df_questions.loc[:,"id_2"] = df_questions["source_2"] + "_" + df_questions["id_2"].astype(str)
df_questions.loc[:,"gold_ids"] = df_questions.apply(lambda row: [row["id_1"], row["id_2"]], axis=1)
print(f"\n  Total questions loaded: {len(df_questions)}")
print(f"  Question types: {df_questions['question_type'].value_counts().to_dict()}")


Step 2: Loading and preparing question datasets...

  Processing: Generic_LeGaulois_Debats
Created dataset with 220 questions from 77 original entries
    Loaded 220 questions

  Processing: Generic_LIntransigeant_Debats
Created dataset with 199 questions from 79 original entries
    Loaded 199 questions

  Processing: BridgeEntity_LIntransigeant_Debats
Created dataset with 152 questions from 79 original entries
    Loaded 152 questions

  Processing: Comparative_LeGaulois_LIntransigeant
Created dataset with 192 questions from 47 original entries
    Loaded 192 questions

  Processing: Generic_LeGaulois_LIntransigeant
Created dataset with 122 questions from 48 original entries
    Loaded 122 questions

  Total questions loaded: 885
  Question types: {'MH - Generic': 541, 'MH - Comparative': 192, 'MH - BridgeEntity': 152}


In [7]:
config = BaseConfig(save_dir=HIPPORAG_DIR_MH,
    llm_name="command-a-03-2025",
    llm_base_url="https://api.cohere.ai/compatibility/v1",
    embedding_model_name="embed-v4.0",
    embedding_base_url="https://api.cohere.ai/compatibility/v1",
    seed=None,
)

rag = HippoRAG(global_config=config)
#rag.index(docs=docs,doc_ids=doc_ids)


In [11]:
output_path = os.path.join(HIPPORAG_DIR_MH, "retrieval_results_MH.jsonl")
from scripts.retrieval_utils import find_gold_ranks

# Load already processed IDs to allow resuming
processed_ids = set()
if os.path.exists(output_path):
    with open(output_path, "r") as f:
        for line in f:
            entry = json.loads(line)
            processed_ids.add(entry["unique_index"])
    print(f"Resuming: {len(processed_ids)} questions already processed.")

for i, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    if row["unique_index"] in processed_ids:
        continue
    retrieval_results = rag.retrieve(queries=[row["question"]], num_to_retrieve=15)
    retrieved_ids = retrieval_results[0].doc_ids
    scores = retrieval_results[0].doc_scores
    gold_ranks = find_gold_ranks(retrieved_ids, row["gold_ids"])
    result = {
        "unique_index": row["unique_index"],
        "question": row["question"],
        "gold_ids": row["gold_ids"],
        "retrieved_ids": retrieved_ids,
        "retrieved_scores": scores.tolist() if hasattr(scores, 'tolist') else list(scores),
        "gold_ranks": gold_ranks
    }
    with open(output_path, "a") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")
    processed_ids.add(row["unique_index"])

5137it [00:00, 9095.55it/s]00<?, ?it/s]
 80%|████████  | 712/885 [44:01<52:16, 18.13s/it]

Error parsing field fact_after_filter: 1 validation error for Fact
  Input should be a valid dictionary or instance of Fact [type=model_type, input_value='{"fact":[[[ ["fact":[[["...##]]}["##]]}["##]]}["##', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/model_type.

		On attempting to parse the value
```
{"fact":[[[ ["fact":[[["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["["[

100%|██████████| 885/885 [53:43<00:00,  3.64s/it]


In [27]:
output_path = os.path.join(HIPPORAG_DIR_MH, "retrieval_results_MH.jsonl")
df_results = pd.DataFrame(read_jsonl_basic(output_path))
df_results = df_results.merge(df_questions[["unique_index", "source_1","source_2"]], on="unique_index", how="left")
results = evaluate_and_print_metrics(df_results, k_values=[3,5,10])


EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


  0%|          | 0/885 [00:00<?, ?it/s]

100%|██████████| 885/885 [00:00<00:00, 24113.49it/s]



Recall Metrics for 885 questions:
  recall@3: 0.317
  recall@5: 0.419
  recall@10: 0.530


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 24534.74it/s]



Accuracy Metrics for 885 questions:
  accuracy@3: 0.116
  accuracy@5: 0.214
  accuracy@10: 0.357


Computing MRR: 100%|██████████| 885/885 [00:00<00:00, 13985.07it/s]


MRR Metrics for 885 questions:
  mrr@3: 0.214
  mrr@5: 0.237
  mrr@10: 0.252


In [25]:
dft = df_results[df_results["source_2"] != "Les Débats"]
print(dft.shape)
results = evaluate_and_print_metrics(dft, k_values=[3,5,10])

(314, 8)

EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


100%|██████████| 314/314 [00:00<00:00, 21698.85it/s]



Recall Metrics for 314 questions:
  recall@3: 0.116
  recall@5: 0.172
  recall@10: 0.247


Computing Accuracy: 100%|██████████| 314/314 [00:00<00:00, 22650.47it/s]



Accuracy Metrics for 314 questions:
  accuracy@3: 0.022
  accuracy@5: 0.045
  accuracy@10: 0.099


Computing MRR: 100%|██████████| 314/314 [00:00<00:00, 13512.80it/s]


MRR Metrics for 314 questions:
  mrr@3: 0.077
  mrr@5: 0.090
  mrr@10: 0.099


In [26]:
dft = df_results[df_results["source_2"] == "Les Débats"]
print(dft.shape)
results = evaluate_and_print_metrics(dft, k_values=[3,5,10])

(571, 8)

EVALUATION RESULTS FOR: DATASET

Computing recall metrics...


100%|██████████| 571/571 [00:00<00:00, 22259.95it/s]



Recall Metrics for 571 questions:
  recall@3: 0.427
  recall@5: 0.554
  recall@10: 0.686


Computing Accuracy: 100%|██████████| 571/571 [00:00<00:00, 23258.69it/s]



Accuracy Metrics for 571 questions:
  accuracy@3: 0.168
  accuracy@5: 0.306
  accuracy@10: 0.499


Computing MRR: 100%|██████████| 571/571 [00:00<00:00, 13669.48it/s]


MRR Metrics for 571 questions:
  mrr@3: 0.289
  mrr@5: 0.318
  mrr@10: 0.336
